# Profiling — Matching Engine (Python vs. Native) and Monte Carlo Runner

`docs/decisions/ADR-005-native-matching-engine-boundary.md` names `MatchingEngine`/`OrderBook`
as the only components ever targeted for a C++ port, on the assumption that they're the
performance-critical hot path. This notebook checks that assumption empirically rather than
leaving it asserted: replays an identical, seeded order sequence through the pure-Python and
native engines and times both, then profiles `MonteCarloRunner.run()` at a realistic batch size
to find where its wall-clock time actually goes.

The first pass through this notebook found a real inefficiency in
`MatchingEngine._match_market` (an unconditional liquidity snapshot nobody read when no
`slippage_model` was configured) — the fix has since been applied to
`exchange/matching/matching_engine.py`, and the numbers below reflect the fixed code. See
`docs/research/02_profiling.md` for the full before/after write-up, including a second,
genuinely surprising finding the fix exposed: pure Python now outperforms the native engine on
this particular order mix.

## 1. Matching engine: pure-Python vs. native, identical order sequence

In [1]:
import time

import numpy as np

from market_sim.core.models import OrderType, Side
from market_sim.exchange.matching import MatchingEngine
from market_sim.exchange.orderbook import Order, OrderBook
from market_sim.exchange.native import NATIVE_AVAILABLE, NativeMatchingEngine, NativeOrderBook

print(f"NATIVE_AVAILABLE = {NATIVE_AVAILABLE}")

NATIVE_AVAILABLE = True


In [2]:
N_ORDERS = 20_000
SEED = 7


def generate_orders(n, seed):
    """Same fuzz-order shape as tests/exchange/test_native_differential.py's
    differential fuzz test: ~80% limit / 20% market, random side, price
    scattered around a mid of 100."""
    rng = np.random.default_rng(seed)
    mid = 100.0
    orders = []
    for i in range(n):
        side = Side.BUY if rng.random() < 0.5 else Side.SELL
        order_type = OrderType.LIMIT if rng.random() < 0.8 else OrderType.MARKET
        quantity = float(rng.integers(1, 20))
        price = (
            round(max(0.01, mid + rng.normal(0, 2.0)), 2)
            if order_type == OrderType.LIMIT
            else None
        )
        orders.append(Order(f"o{i}", side, order_type, quantity, float(i), price))
    return orders


orders = generate_orders(N_ORDERS, SEED)
print(f"Generated {len(orders)} orders")

Generated 20000 orders


In [3]:
def run_python():
    book = OrderBook()
    engine = MatchingEngine()  # no slippage_model -> matches MonteCarloRunner's default
    for i, order in enumerate(orders):
        engine.match(order, book, float(i), i, f"t{i}")


def run_native():
    book = NativeOrderBook()
    engine = NativeMatchingEngine()
    for i, order in enumerate(orders):
        engine.match(order, book, float(i), i, f"t{i}")


# best-of-3, after a warm-up run each
run_python()
run_native()

py_times = []
for _ in range(3):
    t0 = time.perf_counter()
    run_python()
    py_times.append(time.perf_counter() - t0)

native_times = []
for _ in range(3):
    t0 = time.perf_counter()
    run_native()
    native_times.append(time.perf_counter() - t0)

py_best, native_best = min(py_times), min(native_times)
print(f"Python matching engine: best={py_best:.4f}s  all={[f'{t:.4f}' for t in py_times]}")
print(f"Native matching engine: best={native_best:.4f}s  all={[f'{t:.4f}' for t in native_times]}")
print(f"Speedup: {py_best / native_best:.2f}x")

Python matching engine: best=0.0065s  all=['0.0065', '0.0065', '0.0065']
Native matching engine: best=0.0385s  all=['0.0385', '0.0387', '0.0386']
Speedup: 0.17x


**Surprising result**: with the dead liquidity snapshot removed, pure Python is now
**faster** than the native engine on this order mix (`Speedup` above is `py_best / native_best`
< 1, i.e. Python wins) — the opposite of the pre-fix numbers, where the same dead computation
made both engines look artificially close. Native is unchanged (it was never affected by a
Python-only bug); what changed is that Python's real per-order cost is now small enough that the
per-`match()`-call pybind11 boundary-crossing overhead (20,000 independent Python↔C++ round
trips, one per top-level order in this benchmark) dominates the native side instead. This is
workload-shape-dependent, not a blanket "native is slower" result — see
`docs/research/02_profiling.md` for the caveat on what a fairer native-favoring benchmark would
need to look like (fewer, deeper multi-level sweeps per `match()` call, amortizing the
boundary-crossing cost over more fills).

## 2. Where the Python matching engine actually spends its time

`cProfile` over the same 20,000-order replay, sorted by cumulative time.

In [4]:
import cProfile
import pstats

profiler = cProfile.Profile()
profiler.enable()
run_python()
profiler.disable()

stats = pstats.Stats(profiler).sort_stats("cumulative")
stats.print_stats(12)

         117004 function calls in 0.021 seconds

   Ordered by: cumulative time
   List reduced from 32 to 12 due to restriction <12>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        2    0.000    0.000    0.021    0.011 /Users/zeyad/anaconda3/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3490(run_code)
        2    0.000    0.000    0.021    0.011 {built-in method builtins.exec}
        1    0.000    0.000    0.021    0.021 /var/folders/2r/5ggrrhmj5c553v2ckrpvkxxw0000gn/T/ipykernel_81114/3743268583.py:1(<module>)
        1    0.005    0.005    0.021    0.021 /var/folders/2r/5ggrrhmj5c553v2ckrpvkxxw0000gn/T/ipykernel_81114/1936505856.py:1(run_python)
    20000    0.004    0.000    0.016    0.000 /Users/zeyad/Documents/Market simulator/src/market_sim/exchange/matching/matching_engine.py:12(match)
    15958    0.004    0.000    0.011    0.000 /Users/zeyad/Documents/Market simulator/src/market_sim/exchange/matching/matching_engine.py:24(_mat

**Finding (fixed)**: before the fix, almost all matching-engine time (with no `slippage_model`
configured — the `MatchingEngine()` default, and what `MonteCarloRunner` always uses) went to
`OrderBook.bid_liquidity()`/`ask_liquidity()`, called from `MatchingEngine._match_market`.
`_match_market` unconditionally snapshotted `available_liquidity` before checking whether
`self._slippage_model` was even set, even though the snapshot was only read inside the `if
self._slippage_model is not None` branch below it — every market order paid for an O(book
depth) liquidity sum it never used.

That wasn't a guess — `exchange/native/adapter.py`'s `NativeMatchingEngine.match()` already
guarded the equivalent snapshot behind `needs_slippage = self._slippage_model is not None and
incoming.order_type == OrderType.MARKET`, so the same optimization already existed, proven safe,
on the native side of this exact codebase; the pure-Python path just hadn't picked it up. Fixed
by mirroring that guard in `_match_market` — the profile above already reflects the fix: Python
matching-engine time on this 20,000-order replay dropped from ~0.08s to ~0.006s, roughly a
13x reduction, and the remaining time is dominated by real matching logic (`_match_limit`,
`is_filled`, `remaining_quantity`) rather than dead computation.

## 3. MonteCarloRunner at a realistic batch size

In [5]:
import dataclasses

from market_sim.analytics.monte_carlo import MonteCarloRunner
from market_sim.core.config import SimConfig
from market_sim.market.generators import PriceGenerator
from market_sim.strategies import MomentumStrategy

SIM_CONFIG = SimConfig(
    instrument="SIM",
    initial_price=100.0,
    mu=0.05,
    sigma=0.20,
    n_steps=250,
    dt=1 / 252,
    seed=42,
    initial_capital=100_000.0,
)
N_RUNS = 100


def gbm_factory(seed, clock):
    return PriceGenerator(dataclasses.replace(SIM_CONFIG, seed=seed), clock)


def momentum_factory(clock, order_id_factory):
    return MomentumStrategy(
        strategy_id="mc",
        initial_cash=100_000.0,
        clock=clock,
        order_id_factory=order_id_factory,
        lookback=5,
        trade_size=10.0,
    )


def build_runner():
    return MonteCarloRunner(
        price_generator_factory=gbm_factory,
        strategy_factory=momentum_factory,
        n_runs=N_RUNS,
        base_seed=1000,
        initial_cash=100_000.0,
    )


t0 = time.perf_counter()
result = build_runner().run()
elapsed = time.perf_counter() - t0
print(f"n_runs={N_RUNS}, n_steps={SIM_CONFIG.n_steps}")
print(f"elapsed={elapsed:.3f}s, per_run={elapsed / N_RUNS * 1000:.2f}ms")
print(f"mean_pnl={result.mean:.2f}")

n_runs=100, n_steps=250
elapsed=0.417s, per_run=4.17ms
mean_pnl=27933.30


In [6]:
profiler = cProfile.Profile()
profiler.enable()
build_runner().run()
profiler.disable()

stats = pstats.Stats(profiler).sort_stats("cumulative")
stats.print_stats(15)

         3803882 function calls in 1.271 seconds

   Ordered by: cumulative time
   List reduced from 183 to 15 due to restriction <15>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        2    0.000    0.000    1.334    0.667 /Users/zeyad/anaconda3/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3490(run_code)
        2    0.000    0.000    1.334    0.667 {built-in method builtins.exec}
        1    0.001    0.001    1.333    1.333 /Users/zeyad/Documents/Market simulator/src/market_sim/analytics/monte_carlo/monte_carlo_runner.py:103(run)
      100    0.004    0.000    1.332    0.013 /Users/zeyad/Documents/Market simulator/src/market_sim/analytics/monte_carlo/monte_carlo_runner.py:124(_run_once)
      100    0.000    0.000    1.161    0.012 /Users/zeyad/Documents/Market simulator/src/market_sim/core/engine/runtime_engine.py:23(start)
      100    0.035    0.000    1.161    0.012 /Users/zeyad/Documents/Market simulator/src/market_sim/core/engine

**Finding (fixed)**: same dead-liquidity-snapshot fix, same effect at batch scale —
`MonteCarloRunner.run()` for this 100-run batch dropped from 0.672s to 0.413s (~1.6x faster,
4.13ms/run vs. 6.72ms/run before). `MonteCarloRunner` still never has an option to use the
native engine at all (`_run_once` calls `build_exchange()` unconditionally, not
`build_native_exchange()`) — a real architectural gap, not a bug, and per the reversal found in
section 1, wiring native support in wouldn't obviously help this workload shape anyway.

Secondary cost, now proportionally larger since the dominant cost is gone: `uuid.uuid4()` is
called once per `Event` constructed (`events/event.py`'s `event_id` default factory), ~20% of
remaining wall time in this profile. Real, but not pursued further here.

See `docs/research/02_profiling.md` for the full write-up, including the before/after numbers
for both sections.